In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdChemReactions, Draw
import os
import sys
sys.path.append("/home/gridsan/yunsie/git_repo/chemprop")
import chemprop
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
def parity_plot(df_data, df_pred, target_types):
    for target_type in target_types:
        # Set up the figure
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(1, 1, 1)

        # Extract true, predicted, and uncertainty values
        true_list_all = df_data[target_type].to_list()
        pred_list_all = df_pred[target_type].to_list()
        uncertainty_list_all = df_pred[f'{target_type}_ensemble_uncal_var'].to_list()  # Load uncertainty values

        # Remove NaN values
        true_list = []
        pred_list = []
        uncertainty_list = []
        for i in range(len(true_list_all)):
            if not pd.isnull(true_list_all[i]):
                true_list.append(true_list_all[i])
                pred_list.append(pred_list_all[i])
                uncertainty_list.append(uncertainty_list_all[i])

        # Create a density plot using a hexbin
        hb = ax.hexbin(
            true_list,
            pred_list,
            gridsize=100,  # Number of bins for hexagonal tiling
            cmap='plasma',  # Use a colormap to represent density
            mincnt=1  # Minimum count to show a bin
        )

        # Add a colorbar to show the density scale
        cb = plt.colorbar(hb, ax=ax)
        cb.set_label('Density', fontsize=15)

        # Add diagonal line x=y
        axmin = min(min(true_list), min(pred_list)) - 0.1 * (max(true_list) - min(pred_list))
        axmax = max(max(true_list), max(pred_list)) + 0.1 * (max(true_list) - min(pred_list))
        ax.plot([axmin, axmax], [axmin, axmax], 'k--', lw=2, label='Ideal Prediction')

        # Set axis labels and title
        ax.set_xlabel(f'Computed {target_type} (kcal/mol)', fontsize=15)
        ax.set_ylabel(f'Predicted {target_type} (kcal/mol)', fontsize=15)
        ax.set_title(f'Hexbin Plot for {target_type}', fontsize=18)

        # Calculate metrics
        R2 = r2_score(true_list, pred_list)
        mae = mean_absolute_error(true_list, pred_list)
        rmse = mean_squared_error(true_list, pred_list, squared=False)

        # Calculate uncertainty for each metric
        R2_uncertainty = np.std([r2_score(true_list, pred_list + np.random.normal(0, np.sqrt(u), len(pred_list))) for u in uncertainty_list])
        mae_uncertainty = np.std([mean_absolute_error(true_list, pred_list + np.random.normal(0, np.sqrt(u), len(pred_list))) for u in uncertainty_list])
        rmse_uncertainty = np.std([mean_squared_error(true_list, pred_list + np.random.normal(0, np.sqrt(u), len(pred_list)), squared=False) for u in uncertainty_list])

        # Add metrics to the plot with uncertainty
        ax.text(0.05, 0.85, f'$R^2$: {R2:.2f} ± {R2_uncertainty:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')
        ax.text(0.05, 0.95, f'MAE: {mae:.2f} ± {mae_uncertainty:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')
        ax.text(0.05, 0.90, f'RMSE: {rmse:.2f} ± {rmse_uncertainty:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')

        # Customize tick font size
        ax.tick_params(axis='both', which='major', labelsize=12)

        # Show plot
        plt.tight_layout()
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def single_parity_plot(df_data, df_pred):
    # Set up the figure
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(1, 1, 1)

    # Extract true and predicted values
    true_list_all = df_data['GIBBS'].to_list()
    pred_list_all = df_pred['ddGsolv'].to_list()

    # Remove NaN values
    true_list = []
    pred_list = []
    for i in range(len(true_list_all)):
        if not pd.isnull(true_list_all[i]):
            true_list.append(true_list_all[i])
            pred_list.append(pred_list_all[i])

    # Create a density plot using a hexbin
    hb = ax.hexbin(
        true_list,
        pred_list,
        gridsize=100,  # Number of bins for hexagonal tiling
        cmap='plasma',  # Use a colormap to represent density
        mincnt=1  # Minimum count to show a bin
    )

    # Add a colorbar to show the density scale
    cb = plt.colorbar(hb, ax=ax)
    cb.set_label('Density', fontsize=15)

    # Add diagonal line x=y
    axmin = min(min(true_list), min(pred_list)) - 0.1 * (max(true_list) - min(pred_list))
    axmax = max(max(true_list), max(pred_list)) + 0.1 * (max(true_list) - min(pred_list))
    ax.plot([axmin, axmax], [axmin, axmax], 'k--', lw=2, label='Ideal Prediction')

    # Set axis labels and title
    ax.set_xlabel('Computed ∆G_reac (kcal/mol)', fontsize=15)
    ax.set_ylabel('Predicted ∆G_reac (kcal/mol)', fontsize=15)
    ax.set_title('Hexbin Plot for ∆G_reac', fontsize=18)

    # Calculate metrics
    R2 = r2_score(true_list, pred_list)
    mae = mean_absolute_error(true_list, pred_list)
    rmse = mean_squared_error(true_list, pred_list, squared=False)

    # Add metrics to the plot
    ax.text(0.05, 0.85, f'$R^2$: {R2:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.text(0.05, 0.95, f'MAE: {mae:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.text(0.05, 0.90, f'RMSE: {rmse:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')

    # Customize tick font size
    ax.tick_params(axis='both', which='major', labelsize=12)

    # Show plot
    plt.tight_layout()
    plt.show()

In [ ]:
def error_histogram(pred_data, true_data, target_types):
    for target_type in target_types:
        # Extract predicted and computed values
        predicted_values = pred_data[target_type]
        computed_values = true_data[target_type]

        # Calculate the error
        error = predicted_values - computed_values

        # Create a histogram for the error values
        plt.figure(figsize=(5, 4))
        plt.hist(error, bins=100, density=True, color='blue', edgecolor='black')
        plt.title(f"Histogram of {target_type} Error Probability", fontsize=14)
        plt.xlabel(f"{target_type} Error", fontsize=12)
        plt.ylabel("Probability Density", fontsize=12)
        plt.grid(alpha=0.4)

        # Display the plot
        plt.tight_layout()
        plt.show()

In [ ]:
def predicted_histogram(pred_data, target_types):
    for target_type in target_types:
        # Extract predicted and computed values
        predicted_values = pred_data[target_type]

        # Calculate the error
        error = predicted_values

        # Create a histogram for the error values
        plt.figure(figsize=(5, 4))
        plt.hist(error, bins=40, density=True, color='blue', edgecolor='black')
        plt.title(f"Histogram of {target_type} Probability", fontsize=14)
        plt.xlabel(f"{target_type}", fontsize=12)
        plt.ylabel("Probability Density", fontsize=12)
        plt.grid(alpha=0.4)

        # Display the plot
        plt.tight_layout()
        plt.show()

In [ ]:
def computed_histogram(true_data, target_types):
    for target_type in target_types:
        # Extract predicted and computed values
        computed_values = true_data[target_type]

        # Calculate the error
        error = computed_values

        # Create a histogram for the error values
        plt.figure(figsize=(5, 4))
        plt.hist(error, bins=40, density=True, color='blue', edgecolor='black')
        plt.title(f"Histogram of {target_type} Probability", fontsize=14)
        plt.xlabel(f"{target_type}", fontsize=12)
        plt.ylabel("Probability Density", fontsize=12)
        plt.grid(alpha=0.4)

        # Display the plot
        plt.tight_layout()
        plt.show()

In [ ]:
def molwt_histogram(pred_data, target_types):
    pred_data['Weights'] = pred_data[target_types].apply(lambda x: Chem.Descriptors.ExactMolWt(Chem.MolFromSmiles(x)))

    # Create a histogram for the error values
    plt.figure(figsize=(5, 4))
    plt.hist(pred_data['Weights'], bins=15, density=True, color='blue', edgecolor='black')
    plt.title(f"Histogram of Molecular Weight", fontsize=14)
    plt.xlabel(f"Molecular Weight (g/mol)", fontsize=12)
    plt.ylabel("Density", fontsize=12)
    plt.grid(alpha=0.4)

    # Display the plot
    plt.tight_layout()
    plt.show()